# Stage 2 — Baseline models

Naive and seasonal-naive forecasts, evaluated on a held-out 28-day tail.

## Step 1: Train/test split (last 28 days held out)

Splitting by date, not randomly — everything before the cutoff is
"train" (what we'd have known), the last 28 days are "test" (what we're
forecasting). 28 days because that's M5's actual competition horizon.


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

BASE_DIR = Path("/Users/asadkhan.pathan/Documents/New project/m5 forecasting ")
df = pd.read_parquet(BASE_DIR / "data" / "processed" / "sales_long.parquet")
df = df.sort_values(["id", "date"]).reset_index(drop=True)

HORIZON = 28
last_date = df["date"].max()
cutoff_date = last_date - pd.Timedelta(days=HORIZON)

train = df[df["date"] <= cutoff_date]
test = df[df["date"] > cutoff_date]

print("last date in data:", last_date)
print("cutoff date:", cutoff_date)
print("train date range:", train["date"].min(), "to", train["date"].max())
print("test date range:", test["date"].min(), "to", test["date"].max())
print("train rows:", len(train), " test rows:", len(test))
print("test unique days:", test["date"].nunique())  # should be 28


last date in data: 2016-05-22 00:00:00
cutoff date: 2016-04-24 00:00:00
train date range: 2011-01-29 00:00:00 to 2016-04-24 00:00:00
test date range: 2016-04-25 00:00:00 to 2016-05-22 00:00:00
train rows: 58327370  test rows: 853720
test unique days: 28


## Step 2: Naive forecast (flat carry-forward)

For every series, forecast all 28 test days as the last known actual
value from training, held flat. This is the honest naive baseline for a
genuine 28-day-ahead forecast — not a recursive day-by-day repeat, since
real forecasting doesn't get to peek at intermediate actuals.


In [2]:
# train is already sorted by ["id", "date"], so groupby(...).last() correctly
# picks each series' most recent (2016-04-24) value.
last_actual = train.groupby("id", observed=True)["sales"].last()

test = test.merge(last_actual.rename("naive_pred"), on="id", how="left")

print(test[["id", "date", "sales", "naive_pred"]].head(10))
print("any missing naive_pred?", test["naive_pred"].isna().sum())


                              id       date  sales  naive_pred
0  HOBBIES_1_001_CA_1_evaluation 2016-04-25      0           1
1  HOBBIES_1_001_CA_1_evaluation 2016-04-26      0           1
2  HOBBIES_1_001_CA_1_evaluation 2016-04-27      0           1
3  HOBBIES_1_001_CA_1_evaluation 2016-04-28      2           1
4  HOBBIES_1_001_CA_1_evaluation 2016-04-29      0           1
5  HOBBIES_1_001_CA_1_evaluation 2016-04-30      3           1
6  HOBBIES_1_001_CA_1_evaluation 2016-05-01      5           1
7  HOBBIES_1_001_CA_1_evaluation 2016-05-02      0           1
8  HOBBIES_1_001_CA_1_evaluation 2016-05-03      0           1
9  HOBBIES_1_001_CA_1_evaluation 2016-05-04      1           1
any missing naive_pred? 0


## Step 3: Seasonal-naive forecast (weekly cycle)

Forecast each test day using the value from the same weekday one week
back, cycling through the last 7 training days across the 28-day
horizon. Should beat naive here specifically because Stage 1 confirmed
strong weekly seasonality that flat carry-forward ignores.


In [3]:
# Get each series' last 7 training days, and number them 0..6 in date order
# (0 = 7 days before cutoff, 6 = cutoff day itself).
last7 = train.groupby("id", observed=True).tail(7).copy()
last7["pos"] = last7.groupby("id", observed=True).cumcount()

# Reshape to one row per series, one column per weekday-position (wide format).
last7_wide = last7.pivot(index="id", columns="pos", values="sales")

# Number each test day 0..27 per series (0 = first test day), then map to
# a weekday-position 0..6 by taking mod 7 — this cycles the 7-day pattern
# across the 28-day horizon.
test = test.sort_values(["id", "date"])
test["day_offset"] = test.groupby("id", observed=True).cumcount()
test["pos"] = test["day_offset"] % 7

# Melt the wide lookup back to long format and merge onto test by (id, pos).
last7_long = last7_wide.reset_index().melt(id_vars="id", var_name="pos", value_name="seasonal_naive_pred")
test = test.merge(last7_long, on=["id", "pos"], how="left")

print(test[["id", "date", "sales", "naive_pred", "pos", "seasonal_naive_pred"]].head(10))
print("any missing seasonal_naive_pred?", test["seasonal_naive_pred"].isna().sum())


                              id       date  sales  naive_pred pos  \
0  HOBBIES_1_001_CA_1_evaluation 2016-04-25      0           1   0   
1  HOBBIES_1_001_CA_1_evaluation 2016-04-26      0           1   1   
2  HOBBIES_1_001_CA_1_evaluation 2016-04-27      0           1   2   
3  HOBBIES_1_001_CA_1_evaluation 2016-04-28      2           1   3   
4  HOBBIES_1_001_CA_1_evaluation 2016-04-29      0           1   4   
5  HOBBIES_1_001_CA_1_evaluation 2016-04-30      3           1   5   
6  HOBBIES_1_001_CA_1_evaluation 2016-05-01      5           1   6   
7  HOBBIES_1_001_CA_1_evaluation 2016-05-02      0           1   0   
8  HOBBIES_1_001_CA_1_evaluation 2016-05-03      0           1   1   
9  HOBBIES_1_001_CA_1_evaluation 2016-05-04      1           1   2   

   seasonal_naive_pred  
0                    1  
1                    1  
2                    1  
3                    3  
4                    0  
5                    1  
6                    1  
7                    1  
8   

## Step 4: Evaluate naive vs seasonal-naive (plain RMSE/MAE)

Comparing both baselines across all 853,720 test rows. We expect
seasonal-naive to beat naive, since Stage 1 confirmed real weekly
seasonality that flat carry-forward ignores.


In [4]:
def rmse(actual, pred):
    return np.sqrt(np.mean((actual - pred) ** 2))

def mae(actual, pred):
    return np.mean(np.abs(actual - pred))

results = {
    "naive": {
        "RMSE": rmse(test["sales"], test["naive_pred"]),
        "MAE": mae(test["sales"], test["naive_pred"]),
    },
    "seasonal_naive": {
        "RMSE": rmse(test["sales"], test["seasonal_naive_pred"]),
        "MAE": mae(test["sales"], test["seasonal_naive_pred"]),
    },
}

results_df = pd.DataFrame(results).T
print(results_df)


                    RMSE       MAE
naive           2.893557  1.373017
seasonal_naive  2.676895  1.243972
